# Preparação de Dados com PySpark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import month, col
from pyspark.ml.feature import StringIndexer, VectorAssembler, Normalizer, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder.appName("Preparacao").getOrCreate()


## Ler arquivo parquet

In [ ]:
df_video = spark.read.parquet("videos-comments-tratados.snappy.parquet")
df_video.show(5)


## Adicionar coluna Month

In [ ]:
df_video = df_video.withColumn("Month", month(col("Published At")))


## Criar coluna Keyword Index

In [ ]:
indexer = StringIndexer(inputCol="keyword", outputCol="Keyword Index")
df_video = indexer.fit(df_video).transform(df_video)


## Criar vetor Features

In [ ]:
assembler = VectorAssembler(
    inputCols=["Likes", "Views", "Year", "Month", "Keyword Index"],
    outputCol="Features"
)

df_video = assembler.transform(df_video)


## Normalizar Features

In [ ]:
df_video = df_video.dropna(subset=["Features"])

normalizer = Normalizer(inputCol="Features", outputCol="Features Normal")

df_video = normalizer.transform(df_video)


## Aplicar PCA

In [ ]:
pca = PCA(k=1, inputCol="Features Normal", outputCol="Features PCA")

pca_model = pca.fit(df_video)

df_video = pca_model.transform(df_video)


## Separar treino e teste

In [ ]:
train_data, test_data = df_video.randomSplit([0.8, 0.2], seed=42)


## Criar modelo de regressão linear

In [ ]:
lr = LinearRegression(
    featuresCol="Features Normal",
    labelCol="Comments"
)

lr_model = lr.fit(train_data)


## Fazer previsões

In [ ]:
predictions = lr_model.transform(test_data)

predictions.select("Comments", "prediction").show(5)


## Avaliar modelo

In [ ]:
evaluator = RegressionEvaluator(
    labelCol="Comments",
    predictionCol="prediction",
    metricName="rmse"
)

rmse = evaluator.evaluate(predictions)

print("RMSE:", rmse)


## Salvar parquet

In [ ]:
df_video.write.mode("overwrite").parquet("videos-preparados-parquet")
